# ⚡ Udayam Distributed Scraper Worker — Google Colab Edition

Run autonomous Udayam extraction workers in Google Colab for **100% FREE** with **UNLIMITED runtime minutes**.

### ✨ How It Works:
1. Google Colab provides free cloud compute (2 vCPUs, 12 GB RAM).
2. Workers automatically claim tasks from your central Google Sheet.
3. Extracted data is compressed to `.7z` and uploaded directly to Google Drive.
4. Zero impact on GitHub Actions monthly minute quotas.

### Step 1: Clone the Repository
Provide your GitHub Personal Access Token (PAT) to clone your private `udayam-source` repository.
> *Note: You can generate a PAT in GitHub at: **Settings -> Developer settings -> Personal access tokens -> Fine-grained tokens** with `read:contents` scope on `udayam-source`.*

In [ ]:
import os
from getpass import getpass

# Clean up any previous session
!rm -rf udayam-source

github_token = os.getenv('GITHUB_TOKEN') or getpass("Enter your GitHub Personal Access Token: ")
!git clone https://{github_token}@github.com/MaheswaranPalaniselvan/udayam-source.git
%cd udayam-source

### Step 2: Install System & Python Dependencies
Installs 7-Zip, native OCR acceleration libraries, Scrapling, ddddocr, and Playwright Chromium.

In [ ]:
# 1. Install Ubuntu system packages
!apt-get update -qq
!apt-get install -y -qq p7zip-full libopenblas-dev libgomp1 libglib2.0-0 > /dev/null

# 2. Install Python packages
!pip install -q --upgrade pip
!pip install -q "scrapling[all]>=0.4.15" "ddddocr>=1.6.1" "playwright>=1.62.0" "httpx>=0.28.1" "openpyxl>=3.1.5"

# 3. Install Playwright browser engine & system dependencies
!playwright install --with-deps chromium > /dev/null
print("\n All dependencies successfully installed!")

### Step 3: Configure Worker Options
Configure your Google Apps Script Web App URL and run mode:

In [ ]:
# Central Coordinator URL (pre-configured)
os.environ["GOOGLE_APPS_SCRIPT_URL"] = "https://script.google.com/macros/s/AKfycby3tmgH9PtjUhl6XfGrBfgRNZFQZaSLR0N4Dnff9tt9K5qjatyarJy_nnE0ulCtMNm2xA/exec"

# Worker Configuration
MACHINE_NAME = "Colab-Worker-1"      # Identifier shown in Google Sheet & Telegram
CONCURRENCY = 3                     # Parallel browser pages (2 to 4 recommended)
PREFERRED_STATE = ""                # e.g. "TN", "AR", or "" for any available
PREFERRED_DISTRICT = ""             # e.g. "AR-01" or "" for auto
RERUN_TYPE = "none"                 # "none" (normal auto-harvest), "failed", "missing", or "all"

print(f"Configured: Machine={MACHINE_NAME} | Concurrency={CONCURRENCY} | State={PREFERRED_STATE or 'Any'} | District={PREFERRED_DISTRICT or 'Auto'}")

### Step 4: Run Scraper Worker Engine
Executes the autonomous worker. Logs will stream live below, and Google Sheets/Drive will update in real time.

In [ ]:
cmd = [
    "python", "client/app.py",
    "--cli",
    f"--machine={MACHINE_NAME}",
    f"--concurrency={CONCURRENCY}"
]

if PREFERRED_STATE:
    cmd.append(f"--state={PREFERRED_STATE}")
if PREFERRED_DISTRICT:
    cmd.append(f"--district={PREFERRED_DISTRICT}")
if RERUN_TYPE and RERUN_TYPE != "none":
    cmd.append(f"--rerun-type={RERUN_TYPE}")

cmd_str = " ".join(cmd)
print(f"Executing: {cmd_str}\n" + "=" * 60)
!{cmd_str}